In [2]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [3]:
# Fresh kernel — Cell 1
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

In [4]:
spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .getOrCreate()

print("Spark version:", spark.version)  # confirm JVM is alive

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


In [5]:
# Read raw tips data
df_read_tip_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_tip.json")

In [6]:
#df_read_tip_data.show()

In [27]:
tips = df_read_tip_data.select("text")

In [28]:
tips = tips.withColumn(
    "text_clean",
    F.lower(F.trim(F.regexp_replace("text", r"\s+", " ")))
).drop(F.col("text"))

In [ ]:
tips = tips.select(F.col("text_clean")).distinct() #true

In [23]:
df_surrogate_key = tips.withColumn("tips_pk_sk", F.monotonically_increasing_id() + 1)

In [ ]:
test = spark.createDataFrame(
    [("sooooo good!!!!!",), ("yummmmy coffee",), ("cooool place.....",), ("loooove it",)],
    ["text_clean"]
)

test = test.withColumn("fixed", F.regexp_replace("text_clean", r"([a-z])\1{2,}", "$1$1"))
test = test.withColumn("fixed", F.regexp_replace("fixed", r"([!?.,*~-])\1{2,}", "$1"))
test.show(truncate=False)

In [29]:
dup_text = (tips.groupBy("text_clean").count()
                .filter("count > 1")
                .orderBy(F.desc("count")))

dup_text.show(n=dup_text.count(),truncate=False)
print("Different texts that appear more than once:", dup_text.count())

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|text_clean                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [36]:
df_null = tips.filter(F.col("text").contains("N/A"))

In [37]:
df_null.show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|text_clean                                                                                                                                                                                                                                                                                                                                                                                                                      |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------

In [73]:
tips = tips.withColumn("text_raw", F.col("text"))

In [ ]:
tips = tips.withColumn(
    "text_clean",
    F.lower(F.trim(F.regexp_replace("text", r"\s+", " ")))
)

In [ ]:
tips.show(truncate=False)

In [80]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def vader(s: pd.Series) -> pd.Series:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    a = SentimentIntensityAnalyzer()
    return s.fillna("").map(lambda t: a.polarity_scores(t)["compound"])

tips = tips.withColumn("sentiment", vader("text_raw"))

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\sql\pandas\functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\sql\pandas\typehints.py:60: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [82]:
tips = tips.withColumn(
    "sentiment_label",
    F.when(F.col("sentiment") >= 0.05, "positive")
     .when(F.col("sentiment") <= -0.05, "negative")
     .otherwise("neutral")
)

In [83]:
tips.show()

+--------------------+--------------------+--------------------+---------+---------------+
|                text|            text_raw|          text_clean|sentiment|sentiment_label|
+--------------------+--------------------+--------------------+---------+---------------+
|Avengers time wit...|Avengers time wit...|avengers time wit...|      0.0|        neutral|
|They have lots of...|They have lots of...|they have lots of...|   0.4404|       positive|
|It's open even wh...|It's open even wh...|it's open even wh...|      0.0|        neutral|
|Very decent fried...|Very decent fried...|very decent fried...|      0.0|        neutral|
|Appetizers.. plat...|Appetizers.. plat...|appetizers.. plat...|   0.4019|       positive|
|Chili Cup + Singl...|Chili Cup + Singl...|chili cup + singl...|      0.0|        neutral|
|Saturday, Dec 7th...|Saturday, Dec 7th...|saturday, dec 7th...|   0.5562|       positive|
|This is probably ...|This is probably ...|this is probably ...|   0.7579|       positive|

In [84]:
words = tips.select(F.explode(F.split("text_clean", r"[^a-z']+")).alias("word"))
top_words = (words.filter("word != ''")
                  .groupBy("word").count()
                  .orderBy(F.desc("count")))
top_words.show(20)

+-------+------+
|   word| count|
+-------+------+
|    the|437497|
|    and|315926|
|      a|212499|
|     is|176630|
|     to|173140|
|  great|168111|
|    for|138159|
|   food|126223|
|     in|118943|
|      i|110710|
|   good|104601|
|     of|101493|
|    you| 89039|
|service| 85528|
|    was| 83263|
|  place| 80820|
|   this| 78644|
|     on| 78054|
|     it| 77401|
|   they| 73602|
+-------+------+
only showing top 20 rows


In [85]:
tips = (tips
    .withColumn("has_url", F.col("text_raw").rlike(r"https?://|www\."))
    .withColumn("repeat_chars", F.col("text_raw").rlike(r"(.)\1{5,}"))
    .withColumn("is_spam_flag", F.col("has_url") | F.col("repeat_chars")))

In [90]:
tips.orderBy(F.col("repeat_chars").desc()).show(20)

+--------------------+--------------------+--------------------+---------+---------------+-------+------------+------------+
|                text|            text_raw|          text_clean|sentiment|sentiment_label|has_url|repeat_chars|is_spam_flag|
+--------------------+--------------------+--------------------+---------+---------------+-------+------------+------------+
|The brunch was fi...|The brunch was fi...|the brunch was fi...|  -0.5526|       negative|  false|        true|        true|
|Delicious. The or...|Delicious. The or...|delicious. the or...|    0.891|       positive|  false|        true|        true|
|i got roofied her...|i got roofied her...|i got roofied her...|    0.362|       positive|  false|        true|        true|
|Love Ken work!!!!...|Love Ken work!!!!...|love ken work!!!!...|   0.8902|       positive|  false|        true|        true|
|Worst in media......|Worst in media......|worst in media......|  -0.2732|       negative|  false|        true|        true|


In [91]:
tips.select("text_raw", "sentiment", "sentiment_label", "is_spam_flag").show(10, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+---------------+------------+
|text_raw                                                                                                                                                                                                    |sentiment|sentiment_label|is_spam_flag|
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+---------------+------------+
|Avengers time with the ladies.                                                                                                                                                                              |0.0      |neutral        |false       |
|They have lots 